# LGFDR anomaly detection — Swin-B backbone on MVTec AD `transistor`

Unsupervised anomaly detection with **Local-Global Feature Dual Reconstruction (LGFDR)**:

1. A pretrained backbone (`timm`, `features_only=True`) extracts the first two feature stages, which are upsampled,
   concatenated and average-pooled into a feature map **E**.
2. Gaussian noise is injected into **E** during training only.
3. Two convolutional auto-encoders reconstruct the features: a **local** tower (1x1 convs) and a **global**
   tower (strided 3x3 convs + bilinear upsampling).
4. A **Selective Reconstruction Head** (channel-wise softmax over the two towers) fuses them into **U**.
5. The per-pixel L2 distance `||U - E||` is the anomaly map; its pooled maximum is the image-level score.

The model is trained only on defect-free images. Training first fits the reconstruction heads with the backbone
frozen and unfreezes the backbone (lower learning rate) when the loss plateaus; it stops on loss patience.

**Data:** [MVTec AD](https://www.mvtec.com/company/research/datasets/mvtec-ad) (download separately; set `DATA_ROOT` so that `DATA_ROOT/mvtec_ad/<category>/` exists).

In [ ]:
%pip install -q torch torchvision timm scikit-learn matplotlib tqdm

In [ ]:
import os

# Root folder of the dataset(s) used in this notebook.
# Set the DATA_ROOT environment variable or edit the default below.
# On Google Colab you can first mount Drive with:
#   from google.colab import drive; drive.mount('/content/drive')
DATA_ROOT = os.environ.get("DATA_ROOT", "./data")

In [ ]:
images_size_globally = 224
model_name_globally ='swin_base_patch4_window7_224'

In [ ]:
# Updated SwinFeatureExtractor class
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

class SwinFeatureExtractor(nn.Module):
    # Removed the freeze_bn parameter and logic from here
    def __init__(self, model_name=model_name_globally, pretrained=True, pool_kernel=8):
        super().__init__()
        # Ensure pretrained is True for using pre-trained weights
        self.model = timm.create_model(model_name, pretrained=pretrained, features_only=True)
        self.pool_kernel = pool_kernel



        # The freeze_extractor flag in the LGFDRModel will now control
        # requires_grad for ALL parameters in self.model, including BN weights/biases.
        # The train/eval mode of the LGFDRModel will control BN running stats updates.


    def forward(self, x):
        # The parent model's train/eval mode will now control the behavior of BatchNorm layers.
        # No explicit eval() call needed here just for frozen BN anymore.
        features = self.model(x)  # List of feature maps from stages
        features = [f.permute(0, 3, 1, 2) if f.ndim == 4 else f for f in features]
        # Take first two stages only
        if len(features) < 2:
             raise ValueError(f"Feature extractor '{self.model.__class__.__name__}' for model '{self.model.pretrained_cfg['architecture']}' returned less than 2 feature stages ({len(features)}). Cannot proceed.")
        feat1, feat2 = features[:2]  # [(B, C1, H1, W1), (B, C2, H2, W2)]
        # Upsample both to largest (finest) spatial size
        target_size = feat1.shape[-2:]
        # Check if spatial sizes are already the same to avoid unnecessary interpolation
        if feat1.shape[-2:] != feat2.shape[-2:]:
            feat2_up = F.interpolate(feat2, size=target_size, mode='bilinear', align_corners=False)
        else:
            feat2_up = feat2 # No need to interpolate if sizes match

        # Concatenate along channel
        # Ensure number of dimensions match before concatenation
        if feat1.ndim != 4 or feat2_up.ndim != 4:
             raise ValueError(f"Feature stages are not 4D after processing: feat1={feat1.shape}, feat2_up={feat2_up.shape}")

        feat_cat = torch.cat([feat1, feat2_up], dim=1)
        # Average Pooling for false positive reduction (kernel=8)
        feat_pooled = F.avg_pool2d(feat_cat, kernel_size=self.pool_kernel, stride=1, padding=self.pool_kernel//2)
        return feat_pooled  # E: (B, C, H, W)

# You would then instantiate the LGFDRModel passing freeze_extractor=True
# The LGFDRModel's freeze_extractor=True will now set requires_grad=False for all extractor params, *including* BN weights/biases.
# The BN running stats will update during model.train() because you removed the module.eval() call in the extractor's init.
# Example:
# model = LGFDRModel(freeze_extractor=True).to(device) # Use freeze_extractor=True here

In [ ]:
# 2. Noise Injection: Add Gaussian noise during training only
def inject_noise(E, is_train):
    # E: (B, C, H, W)
    if is_train:
        noise = torch.randn_like(E)
        return E + noise
    else:
        return E

In [ ]:
# 3. Dual-Tower CAE Reconstruction Network (IMPROVED)
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=3, stride=1, padding=1, use_bn=True, use_relu=True):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(out_c) if use_bn else nn.Identity()
        self.relu = nn.ReLU(inplace=True) if use_relu else nn.Identity()
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

class LocalCAE(nn.Module):
    # Uses Conv1x1 (standard choice)
    def __init__(self, c):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(c, c, kernel_size=1, padding=0),
            ConvBlock(c, c, kernel_size=1, padding=0),
            ConvBlock(c, c, kernel_size=1, padding=0)
        )
        self.decoder = nn.Sequential(
            ConvBlock(c, c, kernel_size=1, padding=0),
            ConvBlock(c, c, kernel_size=1, padding=0),
            ConvBlock(c, c, kernel_size=1, padding=0, use_relu=False)  # No ReLU in the last block!
        )
        # Optional: output normalization (uncomment if needed)
        # self.out_norm = nn.BatchNorm2d(c)
    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        # out = self.out_norm(out)  # Uncomment if using output normalization
        return out

class GlobalCAE(nn.Module):
    # Uses Conv3x3, stride-2 for down, bilinear up (standard choice)
    def __init__(self, c):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(c, c, kernel_size=3, stride=2, padding=1),
            ConvBlock(c, c, kernel_size=3, stride=2, padding=1),
            ConvBlock(c, c, kernel_size=3, stride=1, padding=1)
        )
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBlock(c, c, kernel_size=3, stride=1, padding=1),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBlock(c, c, kernel_size=3, stride=1, padding=1),
            ConvBlock(c, c, kernel_size=3, stride=1, padding=1, use_relu=False)  # No ReLU in last block!
        )
        # Optional: output normalization (uncomment if needed)
        # self.out_norm = nn.BatchNorm2d(c)
    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        # out = self.out_norm(out)  # Uncomment if using output normalization
        return out

# 4. Selective Reconstruction Head (SRH)
class SRH(nn.Module):
    def __init__(self, c, latent_dim=18):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(2*c, latent_dim),
            nn.ReLU(inplace=True),
            nn.Linear(latent_dim, 2*c)
        )
    def forward(self, L, G):
        # L, G: (B, C, H, W)
        l = L.mean(dim=[2, 3])  # GAP: (B, C)
        g = G.mean(dim=[2, 3])  # (B, C)
        s = torch.cat([l, g], dim=1)  # (B, 2C)
        u = self.mlp(s)  # (B, 2C)
        ul, ug = torch.chunk(u, 2, dim=1)  # (B, C), (B, C)
        # Softmax across local/global for each channel
        w = torch.stack([ul, ug], dim=2)  # (B, C, 2)
        w_soft = torch.softmax(w, dim=2)
        wL = w_soft[:,:,0].unsqueeze(2).unsqueeze(3)  # (B, C, 1, 1)
        wG = w_soft[:,:,1].unsqueeze(2).unsqueeze(3)  # (B, C, 1, 1)
        U = wL * L + wG * G  # (B, C, H, W)
        return U

In [ ]:
# 5. Anomaly Scoring (Pixel-level and Image-level)
def anomaly_maps(U, E, input_size=(images_size_globally, images_size_globally), pool_kernel=32):
    # U, E: (B, C, H, W)
    # Per-pixel L2 norm over channels
    d = torch.norm(U - E, p=2, dim=1)  # (B, H, W)
    # Upsample to input image size
    D = F.interpolate(d.unsqueeze(1), size=input_size, mode='bilinear', align_corners=False).squeeze(1)  # (B, H0, W0)
    # Average pool anomaly map, then max per image (image-level score)
    D_pool = F.avg_pool2d(D, kernel_size=pool_kernel, stride=1, padding=pool_kernel//2)
    S = D_pool.flatten(1).max(dim=1)[0]  # (B,)
    return D, S

In [ ]:
class LGFDRModel(nn.Module):
        def __init__(self, swin_model=model_name_globally, pool_kernel=8, srh_latent=18, freeze_extractor=True): # Add freeze_extractor param
            super().__init__()
            self.extractor = SwinFeatureExtractor(model_name=swin_model, pool_kernel=pool_kernel)
            # We'll infer C after first forward pass
            self.local = None
            self.global_ = None
            self.srh = None
            self.srh_latent = srh_latent
            self.freeze_extractor = freeze_extractor # Store the flag

            if self.freeze_extractor:
                print(f"Freezing parameters of the feature extractor: {swin_model}")
                for param in self.extractor.parameters():
                    param.requires_grad = False

        def build_heads(self, C):
            # Standard: Build heads on first run
            self.local = LocalCAE(C)
            self.global_ = GlobalCAE(C)
            self.srh = SRH(C, latent_dim=self.srh_latent)
            # Move the newly created heads to the same device as the model
            self.local.to(next(self.parameters()).device)
            self.global_.to(next(self.parameters()).device)
            self.srh.to(next(self.parameters()).device)

        def forward(self, x, is_train=True):
            # Ensure extractor is in eval mode if frozen (important for BN layers)
            # Although SwinFeatureExtractor has freeze_bn logic,
            # putting the module itself in eval is safer if you fully freeze it.
            if self.freeze_extractor:
                 self.extractor.eval()
            else:
                 # If not frozen, its mode follows the parent LGFDRModel's mode (train/eval)
                 self.extractor.train(mode=is_train) # Ensure correct mode if not frozen


            E = self.extractor(x)

            if self.local is None:
                # Ensure C is inferred before building heads, even if extractor is frozen
                self.build_heads(E.shape[1])

            # Only inject noise if in training mode AND the extractor is not frozen,
            # or if you want noise on top of frozen features.
            # Common approach: noise on features regardless of freezing
            I = inject_noise(E, is_train) # Noise injection depends only on is_train flag


            L = self.local(I)
            G = self.global_(I)

            # Explicitly resize G to match L's spatial dimensions before passing to SRH
            if L.shape[-2:] != G.shape[-2:]:
                 G_aligned = F.interpolate(G, size=L.shape[-2:], mode='bilinear', align_corners=False)
            else:
                 G_aligned = G

            U = self.srh(L, G_aligned)

            return U, E

In [ ]:
# 7. Training and Inference Step Example (Pixel and Image-level Anomaly Scores)
import torch.optim as optim

# Example usage:
# model = LGFDRModel()
# optimizer = optim.Adam(model.parameters(), lr=1e-4)

def train_step(model, x):
    model.train()
    U, E = model(x, is_train=True)
    # L2 (MSE) loss per pixel, averaged
    loss = ((U - E) ** 2).mean()
    return loss

def inference_step(model, x):
    model.eval()
    with torch.no_grad():
        U, E = model(x, is_train=False)
        D_map, S_score = anomaly_maps(U, E, input_size=(images_size_globally, images_size_globally), pool_kernel=32)
    return D_map, S_score  # Pixel-level map, image-level score

In [ ]:
mvtec_root = os.path.join(DATA_ROOT, "mvtec_ad")  # contains bottle/, transistor/, ...

In [ ]:
# 8a. MVTec AD Dataset Loader and Debug Block (handles _mask.png masks)

import os
import glob
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import matplotlib.pyplot as plt

class MVTecADDataset(Dataset):
    def __init__(self, root, category, split='train', img_size=images_size_globally):
        self.img_paths = []
        self.mask_paths = []
        self.split = split
        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
        ])
        self.mask_transform = T.Compose([
            T.Resize((img_size, img_size), interpolation=Image.NEAREST),
            T.ToTensor(),
        ])
        # Gather image and mask paths
        if split == 'train':
            img_dir = os.path.join(root, category, 'train', 'good')
            self.img_paths = sorted(glob.glob(os.path.join(img_dir, '*.png')) + glob.glob(os.path.join(img_dir, '*.jpg')))
            self.mask_paths = [None] * len(self.img_paths)
        else:  # split == 'test'
            img_dir = os.path.join(root, category, 'test')
            subdirs = sorted(os.listdir(img_dir))
            for sd in subdirs:
                img_files = sorted(glob.glob(os.path.join(img_dir, sd, '*.png')) + glob.glob(os.path.join(img_dir, sd, '*.jpg')))
                self.img_paths.extend(img_files)
                if sd == 'good':
                    self.mask_paths.extend([None]*len(img_files))
                else:
                    # Defective case, has ground_truth
                    mask_dir = os.path.join(root, category, 'ground_truth', sd)
                    for img_path in img_files:
                        basename = os.path.basename(img_path)
                        # Match mask file with _mask.png suffix
                        name_wo_ext = os.path.splitext(basename)[0]
                        mask_file = os.path.join(mask_dir, f"{name_wo_ext}_mask.png")
                        if not os.path.exists(mask_file):
                            print(f"WARNING: No mask found for {img_path}, expected at {mask_file}")
                            mask_file = None
                        self.mask_paths.append(mask_file)

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert('RGB')
        img = self.transform(img)
        # ALWAYS RETURN A MASK TENSOR: if no mask, use zeros
        if self.mask_paths[idx] is not None and os.path.exists(self.mask_paths[idx]):
            mask = Image.open(self.mask_paths[idx]).convert('L')
            mask = self.mask_transform(mask)
            mask = (mask > 0).float()
        else:
            # All zeros mask for normal images
            mask = torch.zeros(1, img.shape[1], img.shape[2])
        return img, mask, self.img_paths[idx]

# ---------- Example usage ----------
category = "transistor"  # Only use categories you have uploaded

train_dataset = MVTecADDataset(mvtec_root, category, split='train')
test_dataset = MVTecADDataset(mvtec_root, category, split='test')
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

# ---------- Debug: Print statistics and sample images ----------
def show_sample_images(data_loader, title="Sample Images", num_samples=5):
    plt.figure(figsize=(10, 4 * num_samples))
    shown = 0
    for img, mask, path in data_loader:
        if shown >= num_samples:
            break
        plt.subplot(num_samples, 2, 2*shown+1)
        plt.imshow(img[0].permute(1, 2, 0))
        plt.title(f"Image: {os.path.basename(path[0])}")
        plt.axis('off')
        plt.subplot(num_samples, 2, 2*shown+2)
        plt.imshow(mask[0][0], cmap='gray')
        plt.title(f"Mask: max={mask.max().item():.1f}")
        plt.axis('off')
        shown += 1
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

print(f"Train images: {len(train_dataset)}")
print(f"Test images: {len(test_dataset)}")

# Print unique test subfolders and sample paths
test_subdirs = sorted(os.listdir(os.path.join(mvtec_root, category, 'test')))
print("Test subfolders:", test_subdirs)
print("First 5 test image paths:", test_dataset.img_paths[:5])

# Count how many test masks are all zeros vs have anomaly
num_anomaly = 0
num_normal = 0
for i in range(len(test_dataset)):
    _, mask, path = test_dataset[i]
    if mask.max() > 0:
        num_anomaly += 1
    else:
        num_normal += 1
print(f"Test set: {num_normal} normal images, {num_anomaly} anomaly images.")

# Show samples from test_loader (edit num_samples as needed)
show_sample_images(test_loader, title="Sample Test Images + Masks", num_samples=4)

Train images: 213
Test images: 100
Test subfolders: ['bent_lead', 'cut_lead', 'damaged_case', 'good', 'misplaced']
First 5 test image paths: ['<DATA_ROOT>/mvtech/transistor/test/bent_lead/000.png', '<DATA_ROOT>/mvtech/transistor/test/bent_lead/001.png', '<DATA_ROOT>/mvtech/transistor/test/bent_lead/002.png', '<DATA_ROOT>/mvtech/transistor/test/bent_lead/003.png', '<DATA_ROOT>/mvtech/transistor/test/bent_lead/004.png']
Test set: 60 normal images, 40 anomaly images.


In [ ]:
# 8b. MVTec AD Training, Testing Loop, and Evaluation (using prepared DataLoader)

from tqdm import tqdm
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
import torch
import torch.optim as optim
import torch.nn as nn # Import nn
from torch.optim import lr_scheduler # Import lr_scheduler

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Instantiate the model. Start with the extractor frozen.
model = LGFDRModel(freeze_extractor=True).to(device)

# --- Trigger build_heads before optimizer initialization ---
# Simulate a forward pass with a dummy tensor to initialize the heads
input_size = next(iter(train_loader))[0].shape
dummy_input = torch.zeros(1, input_size[1], input_size[2], input_size[3]).to(device)
print("Triggering model heads initialization with a dummy input...")
with torch.no_grad():
    _ = model(dummy_input, is_train=False) # Use is_train=False for dummy pass
print("Model heads initialized.")
# --- End of trigger block ---

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")


# --- Dynamic Training Setup ---
# Maximum total epochs to run (as a safeguard)
max_total_epochs = 500  # Increased max epochs since stopping is loss-based

# --- Patience-based Early Stopping and Unfreezing Variables ---
# Note: Removed absolute loss thresholds, relying SOLELY on patience
best_train_loss = float('inf')
patience_counter_stop = 0 # Patience for stopping training
patience_counter_unfreeze = 0 # Patience for unfreezing extractor
patience_value_stop = 20 # How many epochs to wait for IMPROVEMENT before stopping (Increased)
patience_value_unfreeze = 55 # How many epochs to wait for IMPROVEMENT before unfreezing (Increased)
min_delta = 0.0001 # Minimum improvement required to reset patience (for both unfreeze and stop)

print(f"Stop patience value: {patience_value_stop} epochs")
print(f"Unfreeze patience value: {patience_value_unfreeze} epochs")
print(f"Minimum loss improvement delta: {min_delta:.4f}")


base_lr = 1e-1 # Initial learning rate for heads (Keeping user's determined value)
finetune_lr = 1e-4 # Lower learning rate for unfreeze phase (Keeping user's determined value)

print(f"Starting dynamic training (max {max_total_epochs} epochs, unfreeze based on patience, stop based on patience).")

# Initialize optimizer for trainable parameters (initially only heads)
trainable_params_freeze = filter(lambda p: p.requires_grad, model.parameters())
optimizer = torch.optim.Adam(trainable_params_freeze, lr=base_lr)

# Initialize the learning rate scheduler
# Monitor the average training loss
# Reduce LR by a factor (e.g., 0.5) if loss hasn't improved for 'patience' epochs
# min_lr can prevent LR from becoming too small
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, ) # Lowered patience


extractor_frozen = True
print("\n--- Phase: Training with Extractor Frozen ---")

for epoch in range(max_total_epochs):
    model.train()
    total_loss = 0

    # Print current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f"\nEpoch {epoch+1}/{max_total_epochs} | Current LR: {current_lr:.6f}")


    # Keep extractor in eval mode if frozen, set to train if unfrozen
    if extractor_frozen:
        model.extractor.eval()
    else:
        # When unfrozen, the extractor's mode follows the parent LGFDRModel's mode (train/eval)
        model.extractor.train() # Ensure correct mode if not frozen - Removed mode=is_train


    # Use a single tqdm description that updates dynamically or just shows current epoch
    # desc_text = f"Epoch {epoch+1}/{max_total_epochs} ({'Frozen' if extractor_frozen else 'Unfrozen'} Extractor)"
    # for imgs, _, _ in tqdm(train_loader, desc=desc_text):
    # Simpler approach: just show epoch progress
    for imgs, _, _ in tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_total_epochs}"):
        imgs = imgs.to(device)
        loss = train_step(model, imgs)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)

    avg_train_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f}")

    # --- Step the scheduler based on training loss ---
    scheduler.step(avg_train_loss)

    # --- Patience and Early Stopping Logic ---
    # Check for improvement relative to the best loss seen so far
    if avg_train_loss < best_train_loss - min_delta:
        # Improvement found
        best_train_loss = avg_train_loss
        patience_counter_stop = 0 # Reset stop patience
        patience_counter_unfreeze = 0 # Reset unfreeze patience
        print(f"Epoch {epoch+1}: Loss improved to {avg_train_loss:.4f}. Patience counters reset.")
        # Optionally save the best model state here
        # torch.save(model.state_dict(), 'best_model.pth')
    else:
        # No significant improvement
        patience_counter_stop += 1
        if extractor_frozen:
            # Only increment unfreeze patience if extractor is still frozen
             patience_counter_unfreeze += 1
        print(f"Epoch {epoch+1}: No significant improvement. Stop patience: {patience_counter_stop}/{patience_value_stop}, Unfreeze patience: {patience_counter_unfreeze}/{patience_value_unfreeze}")


    # --- Check Patience Condition to Unfreeze ---
    # Unfreeze ONLY if patience for unfreezing is exhausted AND extractor is still frozen
    if extractor_frozen and patience_counter_unfreeze >= patience_value_unfreeze:
        print(f"\n--- Unfreeze patience ({patience_value_unfreeze}) exhausted. Unfreezing extractor parameters. ---")
        extractor_frozen = False
        model.freeze_extractor = False # Update the internal flag
        # --- Reset best_train_loss upon unfreezing ---
        best_train_loss = float('inf') # Reset best loss for the new phase
        print("Best train loss reset for unfrozen phase.")
        # No need to reset patience_counter_unfreeze here, as it will be reset when loss improves

        # Unfreeze the extractor parameters
        print("Unfreezing feature extractor parameters...")
        for name, param in model.extractor.named_parameters():
             if not param.requires_grad: # Only print if it was actually frozen
                 param.requires_grad = True
                 # print(f" - Unfrozen: {name}")


        # Re-initialize the optimizer to include the newly unfrozen parameters
        # AND re-initialize the scheduler with the new optimizer
        trainable_params_unfreeze = filter(lambda p: p.requires_grad, model.parameters())
        num_trainable_unfreeze = sum(1 for _ in trainable_params_unfreeze)
        print(f"Number of trainable parameters after unfreezing: {num_trainable_unfreeze}")
        # Re-create iterator for optimizer
        trainable_params_unfreeze = filter(lambda p: p.requires_grad, model.parameters()) # Need to re-create iterator

        optimizer = torch.optim.Adam(trainable_params_unfreeze, lr=finetune_lr)
        print(f"Optimizer re-initialized with learning rate: {finetune_lr}")

        # Re-initialize the scheduler with the NEW optimizer
        scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, ) # Lowered patience
        print("Scheduler re-initialized with new optimizer.")


        # You might want to print a message indicating the phase has changed
        print("--- Phase: Training with Extractor Unfrozen ---")


    # --- Check Patience Condition to Stop Training ---
    # Stop ONLY if patience for stopping is exhausted
    if patience_counter_stop >= patience_value_stop:
         print(f"\n--- Stop patience ({patience_value_stop}) exhausted. Training loss did not improve significantly for {patience_value_stop} epochs. Stopping training. ---")
         break # Exit the main epoch loop


print("\n--- Training Complete ---")

# --------------- Testing and Evaluation ---------------

model.eval()
all_scores = []
all_labels = []
all_pixel_scores = []
all_pixel_labels = []

with torch.no_grad():
    for imgs, masks, _ in tqdm(test_loader, desc="Testing"):
        imgs = imgs.to(device)
        # Ensure extractor is in eval mode during inference
        model.extractor.eval()
        # Pass is_train=False during inference step
        U, E = model(imgs, is_train=False) # Use the model's forward
        D_map, S_score = anomaly_maps(U, E, input_size=(images_size_globally, images_size_globally), pool_kernel=32)


        # For image-level AUROC: normal=0, anomaly=1 (if mask has any positive pixel, it's anomaly)
        is_anomaly = 1 if masks.max() > 0 else 0
        all_scores.append(S_score.cpu().item())
        all_labels.append(is_anomaly)
        # For pixel-level AUROC/AP
        all_pixel_scores.append(D_map.cpu().numpy().flatten())
        all_pixel_labels.append(masks.cpu().numpy().flatten())

# Compute metrics
all_scores = np.array(all_scores)
all_labels = np.array(all_labels)
print("\n--- Evaluation Results ---")
print("Unique image labels in test set:", set(all_labels))
print("Number of test images:", len(all_labels))
if np.unique(all_labels).size > 1:
    print("Image-level AUROC:", roc_auc_score(all_labels, all_scores))
else:
    print("Image-level AUROC: undefined (only one class present)")

if all_pixel_labels:
    px_scores = np.concatenate(all_pixel_scores)
    px_labels = np.concatenate(all_pixel_labels)
    # Filter out background pixels if necessary (mask=0 often means background)
    # In MVTec, mask=0 is background, mask>0 is anomaly.
    # We want to evaluate AUROC/AP only on pixels that are *not* background (mask > 0 or ground truth anomaly).
    # The anomaly map is computed over the whole image.
    # The standard MVTec pixel AUROC/AP considers all pixels (including background).
    # So, let's keep all flattened pixels unless the task specifies otherwise.
    if np.sum(px_labels) > 0 and np.sum(px_labels) < len(px_labels):
        print("Pixel-level AUROC:", roc_auc_score(px_labels, px_scores))
        print("Pixel-level AP:", average_precision_score(px_labels, px_scores)) # Fixed typo here: px_labels -> px_scores
    else:
        print("Pixel-level AUROC/AP: undefined (no positive mask pixels or only positive pixels)")
else:
    print("No pixel-level masks found.")

# ------------ Instructions for Dataset Setup ------------
# 1. Download the MVTec AD dataset from https://www.mvtec.com/company/research/datasets/mvtec-ad
# 2. Extract the archive under DATA_ROOT/mvtec_ad/
# 3. The directory structure should be: mvtec_ad/category_name/train/good/, mvtec_ad/category_name/test/defect_type/, mvtec_ad/category_name/ground_truth/defect_type/
# 4. Set the DATA_ROOT environment variable (see the setup cell at the top).

Freezing parameters of the feature extractor: swin_base_patch4_window7_224
Triggering model heads initialization with a dummy input...
Model heads initialized.
Total parameters: 95626186
Trainable parameters: 8885010

Trainable Parameter Names and Shapes:
 - local.encoder.0.conv.weight: torch.Size([384, 384, 1, 1]), Num elements: 147456
 - local.encoder.0.bn.weight: torch.Size([384]), Num elements: 384
 - local.encoder.0.bn.bias: torch.Size([384]), Num elements: 384
 - local.encoder.1.conv.weight: torch.Size([384, 384, 1, 1]), Num elements: 147456
 - local.encoder.1.bn.weight: torch.Size([384]), Num elements: 384
... [1033 lines of log omitted] ...

Epoch 249/500 | Current LR: 0.003125
Epoch 249 | Train Loss: 0.0057
Epoch 249: No significant improvement. Stop patience: 19/20, Unfreeze patience: 19/55

Epoch 250/500 | Current LR: 0.003125
Epoch 250 | Train Loss: 0.0057
Epoch 250: No significant improvement. Stop patience: 20/20, Unfreeze patience: 20/55

--- Stop patience (20) exhausted

In [ ]:
# 10. Visualize Original Image, Feature Map E, Reconstructed Feature Map U, Raw Diff, Upsampled Anomaly Map

# Assumes test_loader, model, and device are already defined from previous cells
# and the model has been trained.

model.eval() # Ensure model is in evaluation mode

# Define how many samples of each type you want to visualize
num_normal_samples_to_show = 3
# num_anomaly_samples_to_show = 3 # Commented out as we are visualizing all anomaly images


print(f"\nFetching samples from the test set for visualization ({num_normal_samples_to_show} normal, and all anomaly)...")

# Collect all samples and separate them
all_test_samples = []
normal_samples = []
anomaly_samples = []

# Iterate through the entire test dataset once to categorize samples
for i in tqdm(range(len(test_dataset)), desc="Categorizing test samples"):
    img, mask, path = test_dataset[i]
    sample_info = (img.unsqueeze(0), mask.unsqueeze(0), path) # Add batch dim
    if mask.max() > 0:
        anomaly_samples.append(sample_info)
    else:
        normal_samples.append(sample_info)


print(f"Found {len(normal_samples)} normal samples and {len(anomaly_samples)} anomaly samples in the test set.")

# Select the desired number of samples (take the first few from each list)
# Select 3 normal samples and all anomaly samples
samples_to_visualize = normal_samples[:num_normal_samples_to_show] + anomaly_samples


if not samples_to_visualize:
    print("No samples selected for visualization.")
else:
    actual_samples_count = len(samples_to_visualize)
    print(f"Selected {actual_samples_count} samples for visualization.")

    # --- Calculate Global Min and Max for Heatmaps ---
    print("Calculating global min/max for anomaly heatmaps across all selected samples...")
    all_upsampled_anomaly_maps = []
    all_raw_diff_maps = []

    with torch.no_grad():
        for img, mask, path in tqdm(samples_to_visualize, desc="Processing samples for global scale"):
            img = img.to(device) # Move the image tensor to device

            # Ensure heads are built if this is the very first inference after init
            if model.local is None:
                 model.build_heads(model.extractor(img).shape[1]) # Pass through extractor once to get channel dim

            # Get extracted and reconstructed features
            E = model.extractor(img)
            L = model.local(E)
            G = model.global_(E)

            # Interpolate G to L's size if they are different
            if L.shape[-2:] != G.shape[-2:]:
                 G_interpolated = torch.nn.functional.interpolate(G, size=L.shape[2:], mode='bilinear', align_corners=False)
            else:
                 G_interpolated = G

            U = model.srh(L, G_interpolated)

            # Compute raw diff and upsampled anomaly map
            d_raw = torch.norm(U - E, p=2, dim=1)
            D_map_upsampled, _ = anomaly_maps(U, E, input_size=img.shape[-2:], pool_kernel=32)

            # Store numpy versions
            all_raw_diff_maps.append(d_raw.squeeze(0).cpu().numpy())
            all_upsampled_anomaly_maps.append(D_map_upsampled.squeeze(0).cpu().numpy())

    # Find the global minimum and maximum across all collected maps
    global_raw_diff_min = np.min([np.min(arr) for arr in all_raw_diff_maps]) if all_raw_diff_maps else 0
    global_raw_diff_max = np.max([np.max(arr) for arr in all_raw_diff_maps]) if all_raw_diff_maps else 1
    global_upsampled_min = np.min([np.min(arr) for arr in all_upsampled_anomaly_maps]) if all_upsampled_anomaly_maps else 0
    global_upsampled_max = np.max([np.max(arr) for arr in all_upsampled_anomaly_maps]) if all_upsampled_anomaly_maps else 1


    print(f"Global Raw Diff Min: {global_raw_diff_min:.4f}, Max: {global_raw_diff_max:.4f}")
    print(f"Global Upsampled Anomaly Min: {global_upsampled_min:.4f}, Max: {global_upsampled_max:.4f}")

    # --- Plotting with Global Scale ---

    # Adjust figure size based on the actual number of samples to show
    plt.figure(figsize=(25, 5 * actual_samples_count))

    shown_count = 0
    # Iterate through the selected samples and their pre-calculated maps for plotting
    for (img, mask, path), raw_diff_np, upsampled_map_np in zip(samples_to_visualize, all_raw_diff_maps, all_upsampled_anomaly_maps):

            # Original image needs to be converted to numpy again for plotting
            original_img_np = img.squeeze(0).cpu().permute(1, 2, 0).numpy() # (H, W, C)

            # We need E and U again to calculate average for plotting
            img_on_device = img.to(device)
            with torch.no_grad():
                 E = model.extractor(img_on_device)
                 # Need to rebuild heads here if they weren't built in the previous loop,
                 # although the previous loop should have built them. Redundant check:
                 if model.local is None: model.build_heads(E.shape[1])

                 L = model.local(E)
                 G = model.global_(E)
                 if L.shape[-2:] != G.shape[-2:]:
                      G_interpolated = torch.nn.functional.interpolate(G, size=L.shape[2:], mode='bilinear', align_corners=False)
                 else:
                      G_interpolated = G
                 U = model.srh(L, G_interpolated)
                 _, S_score = anomaly_maps(U, E, input_size=img_on_device.shape[-2:], pool_kernel=32) # Get the score again


            E_avg_np = E.squeeze(0).mean(dim=0).cpu().numpy() # (H_feat, W_feat)
            U_avg_np = U.squeeze(0).mean(dim=0).cpu().numpy() # (H_feat, W_feat)


            # Plotting for this sample
            plt.subplot(actual_samples_count, 5, shown_count * 5 + 1)
            plt.imshow(np.clip(original_img_np, 0, 1))
            # Determine title based on mask
            sample_type = "Anomaly" if mask.max() > 0 else "Normal"
            plt.title(f"{sample_type} Image\n({os.path.basename(path)})")
            plt.axis('off')

            plt.subplot(actual_samples_count, 5, shown_count * 5 + 2)
            plt.imshow(E_avg_np, cmap='gray')
            plt.title("Feature Map E\n(Avg over channels)")
            plt.axis('off')

            plt.subplot(actual_samples_count, 5, shown_count * 5 + 3)
            plt.imshow(U_avg_np, cmap='gray')
            plt.title("Reconstructed Feature Map U\n(Avg over channels)")
            plt.axis('off')

            plt.subplot(actual_samples_count, 5, shown_count * 5 + 4)
            # Plot Raw Diff with GLOBAL vmin/vmax
            im = plt.imshow(raw_diff_np, cmap='hot', vmin=global_raw_diff_min, vmax=global_raw_diff_max)
            plt.title("Raw Diff (||U-E||)\n(Feature Map Size - Global Scale)")
            plt.axis('off')
            plt.colorbar(im, label='L2 Norm')


            plt.subplot(actual_samples_count, 5, shown_count * 5 + 5)
            # Plot Upsampled Anomaly Heatmap with GLOBAL vmin/vmax
            im = plt.imshow(upsampled_map_np, cmap='hot', vmin=global_upsampled_min, vmax=global_upsampled_max)
            plt.title(f"Upsampled Anomaly Heatmap\nScore: {S_score.cpu().item():.4f} (Global Scale)")
            plt.colorbar(im, label='Anomaly Score')
            plt.axis('off')

            shown_count += 1

    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout for potential suptitle
    plt.show()

print("Visualization complete.")

In [ ]:
import matplotlib.pyplot as plt
import torch

def plot_feature_maps_from_loader(feature_extractor, data_loader, device=None):
    """
    Grabs one sample image from the data_loader and plots the first channel of each feature map,
    with correct permutation for [B, H, W, C] --> [B, C, H, W].
    """
    # Get one batch/sample and move to correct device
    img, _, _ = next(iter(data_loader))
    if device is None:
        device = next(feature_extractor.parameters()).device
    img = img.to(device)
    feature_extractor.eval()
    with torch.no_grad():
        features = feature_extractor.model(img)
    if not isinstance(features, (list, tuple)):
        features = [features]
    print(f"Number of feature maps: {len(features)}")
    for i, feat in enumerate(features):
        print(f"Stage {i}: shape = {feat.shape}")
        if feat.ndim == 4:
            # Permute if shape is [B, H, W, C]

            feat = feat.permute(0, 3, 1, 2)
            # Plot first channel of first batch element
            fmap = feat[0, 0].detach().cpu().numpy()
            plt.figure(figsize=(4, 4))
            plt.imshow(fmap, cmap='viridis')
            plt.title(f"Stage {i} | Shape: {feat.shape}")
            plt.axis('off')
            plt.show()
        else:
            print(f"Stage {i} is not 4D, skipping plot.")

# Example usage:
# plot_feature_maps_from_loader(feature_extractor, test_loader)
# Example usage:
# plot_feature_maps_from_loader(feature_extractor, test_loader)
# Make sure you have imported or defined SwinFeatureExtractor in your notebook

# Instantiate the feature extractor
feature_extractor = SwinFeatureExtractor(model_name=model_name_globally)
feature_extractor = feature_extractor.to('cuda' if torch.cuda.is_available() else 'cpu')  # Move to appropriate device

# Now call the plotting function
plot_feature_maps_from_loader(feature_extractor, test_loader)

Number of feature maps: 4
Stage 0: shape = torch.Size([1, 56, 56, 128])


Stage 1: shape = torch.Size([1, 28, 28, 256])


Stage 2: shape = torch.Size([1, 14, 14, 512])


Stage 3: shape = torch.Size([1, 7, 7, 1024])


In [ ]:
# 11. Compare Processed Feature Maps (equivalent to E) from Different Pre-trained Models

import torch
import timm
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader # Ensure Dataset is imported
import torch.nn as nn
import torch.nn.functional as F
import math # Import math for calculating grid size

# Define the device (GPU or CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")


# Assumes MVTecADDataset and SwinFeatureExtractor class are defined in previous cells
# Assumes mvtec_root is defined in previous cells
'''
'''
# --- Select a single model to test ---
# You can change this name to any valid timm model that works with your desired input size
model_base_name_to_test = 'resnet18' # Change this to your desired model

# --- Set the desired input size for THIS visualization ---
# This will be used to create a temporary dataset/dataloader
img_size_for_testing = 256 # <<< CHANGE THIS VALUE to your desired input size for visualization


# --- Create a temporary dataset and dataloader for fetching sample image at the desired size ---
print(f"Creating temporary test dataset with image size {img_size_for_testing}x{img_size_for_testing}...")

# Reuse the existing MVTecADDataset class but configure it with the specific test size
temp_test_dataset = MVTecADDataset(mvtec_root, category, split='test', img_size=img_size_for_testing) # Use img_size_for_testing here

# Create a DataLoader for this temporary dataset
# Use a batch size of 1 and shuffle=False as we just need one sample
temp_test_loader = DataLoader(temp_test_dataset, batch_size=1, shuffle=False, num_workers=0) # num_workers=0 often safer for single sample


# Get one sample anomaly image from the temporary test loader if possible
sample_img = None
sample_path = None
found_anomaly = False

print(f"Fetching a sample ANOMALY image from the temporary test set...")

with torch.no_grad():
    # Iterate through the temporary loader
    for imgs, masks, paths in temp_test_loader:
        if masks.max() > 0: # Check if mask has any positive pixels
            sample_img = imgs[0].unsqueeze(0) # Take the first image from this batch
            sample_path = paths[0]
            found_anomaly = True
            break # Found an anomaly sample

# If no anomaly found after checking the entire loader, just take the very first image
if not found_anomaly and len(temp_test_dataset) > 0:
     print("Could not find an anomaly sample. Using the first normal image from the temporary test set.")
     sample_img, _, sample_path = temp_test_dataset[0]
     sample_img = sample_img.unsqueeze(0) # Add batch dimension
     sample_path = temp_test_dataset.img_paths[0] # Get the actual path string
elif sample_img is None:
    print("Temporary test loader is empty or could not load any sample image. Cannot perform visualization.")


if sample_img is not None:
    sample_img_processed = sample_img.to(device) # Move the sample to device
    print(f"Sample image shape loaded from temporary loader: {sample_img_processed.shape}")

    print(f"\nProcessing features for model: {model_base_name_to_test}")

    try:
         # Instantiate your SwinFeatureExtractor with the current model name
         # Note: The extractor itself doesn't have an 'img_size' param, it adapts to input size
         extractor = SwinFeatureExtractor(model_name=model_base_name_to_test, pretrained=True, pool_kernel=8).to(device)
         extractor.eval() # Set to evaluation mode

         with torch.no_grad():
              # Pass the sample image through your extractor logic
              E_equivalent = extractor(sample_img_processed)

         print(f"  Output feature shape: {E_equivalent.shape}")

         # Check if the output is 4D (B, C, H, W) as expected
         if E_equivalent.ndim == 4:
             feature_map = E_equivalent.squeeze(0).cpu() # Remove batch dim, move to CPU
             num_channels = feature_map.shape[0]
             feat_height, feat_width = feature_map.shape[1:]

             print(f"  Visualizing {num_channels} channels ({feat_height}x{feat_width})...")

             # Calculate grid dimensions
             # Try to make it roughly square, or wider if many channels
             grid_cols = math.ceil(math.sqrt(num_channels))
             grid_rows = math.ceil(num_channels / grid_cols)

             # Prepare figure for the grid of channels
             # Adjust figure size based on number of channels and subplot size
             fig_width = grid_cols * 2.5 # Adjust multiplier as needed
             fig_height = grid_rows * 2.5 # Adjust multiplier as needed
             fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(fig_width, fig_height))
             # Flatten axes array for easy iteration if it's a multi-row/col grid
             axes = axes.flatten() if grid_rows > 1 or grid_cols > 1 else [axes]


             fig.suptitle(f"Processed Feature Map Channels for {model_base_name_to_test}\nSample: {os.path.basename(sample_path)} (Input size: {img_size_for_testing}x{img_size_for_testing})", fontsize=14)


             # Plot each channel
             for channel_idx in range(num_channels):
                 if channel_idx < len(axes): # Safety check for grid size
                     ax = axes[channel_idx]
                     # Use 'gray' cmap for single channel visualization
                     im = ax.imshow(feature_map[channel_idx].numpy(), cmap='gray')
                     ax.set_title(f"Channel {channel_idx}")
                     ax.axis('off')
                     # Optionally add a colorbar for each channel, but might clutter
                     # plt.colorbar(im, ax=ax)
                 else:
                     print(f"Warning: More channels ({num_channels}) than subplots ({len(axes)}). Skipping remaining channels.")
                     break # Stop plotting if we run out of axes


             # Hide any unused subplots if num_channels doesn't fill the grid
             for i in range(num_channels, len(axes)):
                 axes[i].axis('off')


             plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout for suptitle
             plt.show()

         else:
             # Handle unexpected output dimensions from the extractor
             print(f"  Unexpected processed feature shape for {model_base_name_to_test}: {E_equivalent.shape}. Cannot visualize channels.")


    except Exception as e:
         print(f"  Error processing features for {model_base_name_to_test}: {e}")


print("Processed feature map channel visualization complete for the selected model.")